# Klusterisasi Kinerja Siswa dengan Fuzzy C-Means + PCA
### Notebook pendamping Laporan Proyek — Mata Kuliah Teknologi Logika Fuzzy

Notebook ini mereproduksi **seluruh angka, tabel, dan gambar** pada laporan resmi (BAB III–IV) untuk kombinasi **Fuzzy C-Means (FCM) + Principal Component Analysis (PCA)**. Algoritma FCM ditulis mandiri dengan NumPy; PCA, standardisasi, dan metrik evaluasi memakai scikit-learn. Perbandingan dengan FCM tanpa PCA tidak dilakukan.

**Evaluasi sistem terdiri atas empat komponen:** (1) uji kelayakan PCA *sebelum ekstraksi* dengan **KMO** dan **Bartlett's Test of Sphericity**, (2) **Explained Variance** *setelah ekstraksi*, (3) **Silhouette Score**, dan (4) **Fuzzy Partition Coefficient**.

| No | Bagian notebook | Padanan di laporan |
|---|---|---|
| 0 | Persiapan dan parameter | 4.1, Tabel 4.1 |
| 1 | Load data dan validasi | 3.2, 4.3.1, Tabel 4.6 |
| 2 | Deskripsi variabel | Tabel 3.2–3.3 |
| 3 | **Evaluasi 1** — Uji kelayakan PCA sebelum ekstraksi (KMO, Bartlett) | Gambar 3.1, 4.6.2, Tabel 4.7–4.8 |
| 4 | Implementasi FCM mandiri | 2.2, 4.3.3 |
| 5 | Contoh perhitungan manual | 4.4, Tabel 4.3–4.5, Gambar 4.1 |
| 6 | Standardisasi dan PCA, **Evaluasi 2** — Explained variance setelah ekstraksi | 4.6.3, Tabel 4.9, Gambar 4.2 |
| 7 | Pemilihan jumlah cluster | 4.6.4, Tabel 4.10, Gambar 4.3 |
| 8 | Sensitivitas parameter | 4.6.5, Tabel 4.11–4.12, Gambar 4.4 |
| 9 | Klasterisasi final dan penamaan cluster | 4.6.6, Tabel 4.13–4.14, Gambar 4.5–4.7 |
| 10 | **Evaluasi 3 dan 4** — Silhouette Score dan Fuzzy Partition Coefficient, ringkasan evaluasi | 4.6.7, Tabel 4.15, Gambar 4.8–4.9 |
| 11 | Prediksi siswa baru | 4.6.8, Tabel 4.16 |
| 12 | Interpretasi dan kesimpulan | 4.7, BAB V |

## 0. Persiapan dan Parameter
Seluruh parameter di sel berikut dapat diubah. Nilai bawaan sama dengan aplikasi Streamlit dan laporan.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
DK, TL, AM, PU, RD, GR = "#0B3C49", "#1B998B", "#F4A259", "#7B5EA7", "#C8553D", "#5B6B70"
CL = [TL, AM, PU, RD]            # warna cluster

# ====== PARAMETER ======
CSV_PATH      = "student-mat.csv"   # lokasi dataset (dicari otomatis bila tidak ditemukan)
M_FUZZ        = 1.5                 # fuzzifier m (>1). m = 2 degenerat pada data ini (lihat bagian 8)
MAX_ITER      = 500                 # iterasi maksimum FCM
TOL           = 1e-5                # toleransi konvergensi
N_INIT        = 5                   # jumlah inisialisasi acak (dipilih J terkecil)
VAR_THRESHOLD = 0.80                # ambang explained variance kumulatif PCA
K_MIN, K_MAX  = 2, 8                # rentang kandidat jumlah cluster
K_MODE        = "auto"              # "auto" (Silhouette tertinggi) atau "manual"
K_MANUAL      = 3                   # dipakai bila K_MODE = "manual"
SEED          = 42                  # random seed (reprodusibilitas)

for p in [CSV_PATH, "fcm_pca_app/student-mat.csv", "/mnt/user-data/uploads/student-mat.csv"]:
    if os.path.exists(p):
        CSV_PATH = p; break
print("Dataset :", CSV_PATH)
print("Parameter: m =", M_FUZZ, "| maks iterasi =", MAX_ITER, "| toleransi =", TOL, "| n_init =", N_INIT,
      "| ambang varians =", VAR_THRESHOLD, "| k =", K_MIN, "-", K_MAX, "| seed =", SEED)

## 1. Load Data dan Validasi Masukan
Fungsi `validate_dataset` memeriksa kelengkapan 16 kolom wajib, mengonversi nilai ke numerik, membuang baris kosong atau di luar rentang valid, dan menolak data dengan kurang dari 30 baris valid. Fungsi yang sama dipakai aplikasi Streamlit.

In [ ]:
FEATURES = ["age", "Medu", "Fedu", "traveltime", "studytime", "failures",
            "famrel", "freetime", "goout", "Dalc", "Walc", "health",
            "absences", "G1", "G2", "G3"]

# variabel: (nama, min, max, keterangan skala)
FEATURE_INFO = {
    "age":        ("Usia", 15, 22, "Usia siswa (tahun)."),
    "Medu":       ("Pendidikan ibu", 0, 4, "0 = tidak sekolah, 1 = SD, 2 = SMP, 3 = SMA, 4 = pendidikan tinggi."),
    "Fedu":       ("Pendidikan ayah", 0, 4, "Sama dengan skala pendidikan ibu."),
    "traveltime": ("Waktu tempuh ke sekolah", 1, 4, "1 = <15 mnt, 2 = 15-30 mnt, 3 = 30-60 mnt, 4 = >60 mnt."),
    "studytime":  ("Waktu belajar mingguan", 1, 4, "1 = <2 jam, 2 = 2-5 jam, 3 = 5-10 jam, 4 = >10 jam."),
    "failures":   ("Jumlah kegagalan kelas", 0, 4, "Jumlah kegagalan kelas sebelumnya."),
    "famrel":     ("Kualitas hubungan keluarga", 1, 5, "1 = sangat buruk ... 5 = sangat baik."),
    "freetime":   ("Waktu luang setelah sekolah", 1, 5, "1 = sangat sedikit ... 5 = sangat banyak."),
    "goout":      ("Frekuensi bergaul", 1, 5, "1 = sangat jarang ... 5 = sangat sering."),
    "Dalc":       ("Konsumsi alkohol hari kerja", 1, 5, "1 = sangat rendah ... 5 = sangat tinggi."),
    "Walc":       ("Konsumsi alkohol akhir pekan", 1, 5, "1 = sangat rendah ... 5 = sangat tinggi."),
    "health":     ("Status kesehatan", 1, 5, "1 = sangat buruk ... 5 = sangat baik."),
    "absences":   ("Jumlah absensi", 0, 93, "Jumlah ketidakhadiran di sekolah."),
    "G1":         ("Nilai periode 1", 0, 20, "Nilai periode pertama."),
    "G2":         ("Nilai periode 2", 0, 20, "Nilai periode kedua."),
    "G3":         ("Nilai akhir", 0, 20, "Nilai akhir."),
}

# nilai bawaan formulir input manual pada aplikasi (profil tengah)
FORM_DEFAULT = {"age": 17, "Medu": 2, "Fedu": 2, "traveltime": 1, "studytime": 2, "failures": 0, "famrel": 4, "freetime": 3,
                "goout": 3, "Dalc": 1, "Walc": 1, "health": 3, "absences": 4, "G1": 10, "G2": 10, "G3": 10}

def read_dataset(path):
    """Baca CSV; separator (; atau ,) dideteksi otomatis."""
    d = pd.read_csv(path, sep=None, engine="python")
    d.columns = [str(c).strip().strip('"') for c in d.columns]
    return d

def validate_dataset(d, min_rows=30):
    """Mengembalikan (data_bersih, daftar_pesan, ringkasan)."""
    msgs = []
    missing = [c for c in FEATURES if c not in d.columns]
    if missing:
        msgs.append(("error", f"Kolom wajib tidak ditemukan: {', '.join(missing)}."))
        return None, msgs, {}
    d = d.copy()
    for c in FEATURES:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    n0 = len(d)
    bad_na = d[FEATURES].isna().any(axis=1)
    bad_rng = pd.Series(False, index=d.index)
    for c in FEATURES:
        _, lo, hi, _ = FEATURE_INFO[c]
        bad_rng |= (d[c] < lo) | (d[c] > hi)
    bad = bad_na | bad_rng
    if bad_na.sum():
        msgs.append(("warning", f"{int(bad_na.sum())} baris memiliki nilai kosong/non-numerik pada fitur dan dibuang."))
    if (bad_rng & ~bad_na).sum():
        msgs.append(("warning", f"{int((bad_rng & ~bad_na).sum())} baris memiliki nilai di luar rentang valid dan dibuang."))
    clean = d.loc[~bad].reset_index(drop=True)
    if len(clean) < min_rows:
        msgs.append(("error", f"Data valid hanya {len(clean)} baris; minimal {min_rows} baris dibutuhkan."))
        return None, msgs, {}
    if not bad.any():
        msgs.append(("success", "Seluruh baris lolos validasi (kolom lengkap, numerik, dan dalam rentang)."))
    return clean, msgs, {"baris_awal": n0, "baris_valid": len(clean), "fitur": len(FEATURES)}

def validate_single(values):
    """Validasi input manual satu siswa."""
    errs = []
    for c, v in values.items():
        lbl, lo, hi, _ = FEATURE_INFO[c]
        if v is None or (isinstance(v, float) and np.isnan(v)):
            errs.append(f"{lbl} ({c}) kosong.")
        elif not (lo <= v <= hi):
            errs.append(f"{lbl} ({c}) harus di antara {lo} dan {hi}.")
    return errs

raw = read_dataset(CSV_PATH)
df, msgs, info = validate_dataset(raw)
for lvl, t in msgs:
    print(f"[{lvl.upper()}] {t}")
print(info)
print("Missing value pada 16 fitur :", int(df[FEATURES].isna().sum().sum()))
df[FEATURES].head(10)      # Lampiran D laporan

### 1.1 Pengujian validasi masukan (Tabel 4.6)
Dataset sengaja dimodifikasi untuk menguji bahwa data tidak valid ditolak atau dibersihkan.

In [ ]:
tests = []
d1 = raw.drop(columns=["G3"]); c1, m1, _ = validate_dataset(d1)
tests.append(("Kolom wajib hilang (G3 dihapus)", m1[0][1], "Ditolak (sesuai harapan)" if c1 is None else "GAGAL"))

d2 = raw.copy(); d2.loc[0:4, "age"] = 40; c2, m2, _ = validate_dataset(d2)
tests.append(("5 baris age = 40 (di luar rentang)", " | ".join(t for _, t in m2), f"Baris dibuang; {len(c2)} baris tersisa"))

d3 = raw.copy(); d3["G1"] = d3["G1"].astype(object); d3.loc[0:2, "G1"] = "abc"; c3, m3, _ = validate_dataset(d3)
tests.append(("3 baris G1 non-numerik", " | ".join(t for _, t in m3), f"Baris dibuang; {len(c3)} baris tersisa"))

c4, m4, _ = validate_dataset(raw.head(10))
tests.append(("Hanya 10 baris", m4[-1][1], "Ditolak (sesuai harapan)" if c4 is None else "GAGAL"))

c5, m5, _ = validate_dataset(raw)
tests.append(("Dataset asli", m5[-1][1], f"{len(c5)} baris diterima"))

bad = {c: (FEATURE_INFO[c][1] + FEATURE_INFO[c][2]) // 2 for c in FEATURES}; bad["G3"] = 25; bad["health"] = 0
tests.append(("Input manual: health = 0 dan G3 = 25", " ".join(validate_single(bad)), "Ditolak (sesuai harapan)"))
good = {c: float(v) for c, v in FORM_DEFAULT.items()}
tests.append(("Input manual: seluruh isian bawaan", "Tidak ada pesan kesalahan" if not validate_single(good) else "ada pesan", "Diterima (sesuai harapan)"))

tab_validasi = pd.DataFrame(tests, columns=["Skenario", "Pesan sistem", "Hasil"]); tab_validasi.index += 1
display(tab_validasi)

## 2. Deskripsi Variabel dan Statistik Deskriptif (Tabel 3.2–3.3)
Hanya 16 atribut numerik yang dipakai. Atribut kategorikal (`school`, `sex`, `Mjob`, dll.) tidak dipakai karena FCM dan PCA berbasis jarak Euclidean.

In [ ]:
N = len(df)
tab_var = pd.DataFrame({
    "Nama": [FEATURE_INFO[c][0] for c in FEATURES],
    "Rentang": [f"{FEATURE_INFO[c][1]}-{FEATURE_INFO[c][2]}" for c in FEATURES],
    "Rata-rata": df[FEATURES].mean().values, "Std": df[FEATURES].std().values,
    "Min": df[FEATURES].min().values, "Maks": df[FEATURES].max().values,
    "Keterangan skala": [FEATURE_INFO[c][3] for c in FEATURES]}, index=FEATURES)
display(tab_var.round(2))
print(f"Jumlah siswa            : {N}")
print(f"Rata-rata nilai akhir G3: {df['G3'].mean():.2f}")
print(f"Siswa dengan G3 = 0     : {(df['G3'] == 0).sum()}")
print(f"Siswa dengan G3 < 10    : {(df['G3'] < 10).mean():.1%}")

## 3. Evaluasi 1 — Uji Kelayakan PCA Sebelum Ekstraksi (Gambar 3.1, Tabel 4.7–4.8)
Korelasi antar fitur menjadi alasan memakai PCA. Sebelum PCA dijalankan, kelayakan data diuji dengan dua uji:

- **Bartlett's Test of Sphericity**: menguji $H_0$ bahwa matriks korelasi $R$ adalah matriks identitas (antar-fitur tidak berkorelasi). $\chi^2=-\left(n-1-\frac{2p+5}{6}\right)\ln|R|$ dengan derajat bebas $p(p-1)/2$. $H_0$ ditolak bila $p<0{,}05$ sehingga PCA bermanfaat.
- **KMO (Kaiser-Meyer-Olkin)**: kecukupan sampel, $KMO=\dfrac{\sum\sum r_{ij}^2}{\sum\sum r_{ij}^2+\sum\sum q_{ij}^2}$ dengan $q_{ij}$ korelasi parsial. Kriteria: ≥ 0,8 sangat baik; ≥ 0,7 baik; ≥ 0,6 cukup; < 0,5 buruk.

In [ ]:
C = df[FEATURES].corr()
fig, ax = plt.subplots(figsize=(6.6, 5.6))
im = ax.imshow(C.values, cmap="RdBu_r", vmin=-1, vmax=1); ax.grid(False)
ax.set_xticks(range(16)); ax.set_xticklabels(FEATURES, rotation=90, fontsize=8)
ax.set_yticks(range(16)); ax.set_yticklabels(FEATURES, fontsize=8)
plt.colorbar(im, label="Korelasi Pearson"); ax.set_title("Gambar 3.1  Matriks korelasi 16 fitur numerik")
plt.tight_layout(); plt.show()

pairs = C.where(np.triu(np.ones(C.shape), 1).astype(bool)).stack().reset_index()
pairs.columns = ["Fitur A", "Fitur B", "Korelasi"]
print("Sepuluh pasangan fitur dengan korelasi terkuat:")
display(pairs.reindex(pairs["Korelasi"].abs().sort_values(ascending=False).index).head(10).round(3).reset_index(drop=True))

In [ ]:
def pca_suitability(d):
    """Bartlett sphericity dan KMO."""
    X_ = d[FEATURES].astype(float); n, p = X_.shape
    R_ = np.corrcoef(X_.values, rowvar=False)
    chi2 = -(n - 1 - (2 * p + 5) / 6) * np.linalg.slogdet(R_)[1]
    dof = p * (p - 1) / 2
    pval = float(stats.chi2.sf(chi2, dof))
    Rinv = np.linalg.pinv(R_); dd = np.sqrt(np.outer(np.diag(Rinv), np.diag(Rinv)))
    partial = -Rinv / dd
    np.fill_diagonal(R_, 0); np.fill_diagonal(partial, 0)
    kmo_per = (R_ ** 2).sum(0) / ((R_ ** 2).sum(0) + (partial ** 2).sum(0))
    kmo = float((R_ ** 2).sum() / ((R_ ** 2).sum() + (partial ** 2).sum()))
    return dict(chi2=float(chi2), dof=int(dof), p=pval, kmo=kmo, kmo_per=pd.Series(kmo_per, index=FEATURES))

suit = pca_suitability(df)
kmo_cat = ("sangat baik" if suit["kmo"] >= .8 else "baik" if suit["kmo"] >= .7 else "cukup" if suit["kmo"] >= .6
           else "kurang" if suit["kmo"] >= .5 else "buruk")
print(f"KMO              = {suit['kmo']:.3f}  ({kmo_cat})")
print(f"Bartlett chi2    = {suit['chi2']:.1f}  (db = {suit['dof']}, p = {suit['p']:.2e})")
layak = suit["p"] < .05 and suit["kmo"] >= .6
print("Kesimpulan       :", "data LAYAK untuk PCA" if layak else "kelayakan PCA kurang")
tab_kmo = pd.DataFrame([("KMO keseluruhan", f"{suit['kmo']:.3f}", f"Kategori: {kmo_cat} (>= 0,6 layak)"),
                        ("Bartlett chi-kuadrat", f"{suit['chi2']:.1f}", f"Derajat bebas = {suit['dof']}"),
                        ("Bartlett p-value", "< 0,001" if suit["p"] < .001 else f"{suit['p']:.4f}", "H0 ditolak: ada korelasi antar-fitur" if suit["p"] < .05 else "H0 diterima")],
                       columns=["Ukuran", "Nilai", "Interpretasi"]).set_index("Ukuran")
display(tab_kmo)
display(suit["kmo_per"].rename("KMO per fitur").round(3).to_frame().T)

## 4. Implementasi Fuzzy C-Means (NumPy)
Fungsi objektif: $J_m=\sum_{i=1}^{N}\sum_{j=1}^{c} u_{ij}^m \, \|x_i - v_j\|^2$  
Pusat cluster: $v_j=\dfrac{\sum_i u_{ij}^m x_i}{\sum_i u_{ij}^m}$  
Keanggotaan: $u_{ij}=\dfrac{1}{\sum_{k=1}^{c}\left(\dfrac{d_{ij}}{d_{ik}}\right)^{2/(m-1)}}$, dengan $d_{ij}=\|x_i-v_j\|$  
Iterasi berhenti saat $\max|U^{(t+1)}-U^{(t)}|<\varepsilon$ atau mencapai iterasi maksimum. FCM dijalankan `N_INIT` kali dan hasil dengan $J$ terkecil dipilih.

Ketegasan partisi fuzzy diukur dengan Fuzzy Partition Coefficient $PC=\frac1N\sum\sum u_{ij}^2$ (batas bawah $1/c$).

In [ ]:
def _dist(X, V):
    """Jarak Euclidean (N x c) secara vectorized."""
    d2 = (X ** 2).sum(1)[:, None] - 2 * X @ V.T + (V ** 2).sum(1)[None, :]
    return np.sqrt(np.maximum(d2, 1e-12))

def memberships(X, V, m):
    """Derajat keanggotaan data X terhadap pusat V (Persamaan u_ij)."""
    D = _dist(X, V)
    inv = D ** (-2.0 / (m - 1))
    return inv / inv.sum(1, keepdims=True)

def fcm_single(X, c, m, max_iter, tol, rng):
    N = X.shape[0]
    U = rng.random((N, c)); U /= U.sum(1, keepdims=True)      # inisialisasi U acak
    hist = []
    for it in range(1, max_iter + 1):
        U_old = U
        Um = U ** m
        V = (Um.T @ X) / Um.sum(0)[:, None]                   # pusat cluster v_j
        D = _dist(X, V)                                       # jarak d_ij
        inv = D ** (-2.0 / (m - 1))
        U = inv / inv.sum(1, keepdims=True)                   # keanggotaan u_ij
        hist.append(float(((U ** m) * D ** 2).sum()))         # fungsi objektif J
        if np.abs(U - U_old).max() < tol:                     # kriteria konvergensi
            break
    J = float(((U ** m) * _dist(X, V) ** 2).sum())
    return U, V, J, it, hist

def fcm(X, c, m=1.5, max_iter=500, tol=1e-5, n_init=5, seed=42):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        r = fcm_single(X, c, m, max_iter, tol, rng)
        if best is None or r[2] < best[2]:
            best = r
    U, V, J, n_it, hist = best
    return {"U": U, "centers": V, "J": J, "n_iter": n_it, "hist": hist, "labels": U.argmax(1)}

def partition_coefficient(U): return float((U ** 2).sum() / U.shape[0])
def is_degenerate(U, tol=1e-3): return partition_coefficient(U) < 1.0 / U.shape[1] + tol
print("Fungsi FCM siap dipakai.")

## 5. Contoh Perhitungan Manual (BAB IV 4.4)
Untuk memperlihatkan pemahaman algoritma, alur **standardisasi → PCA → FCM** dihitung pada lima data sintetis dua fitur (bukan bagian dari dataset penelitian). Dipilih 1 komponen PCA (PC1), kemudian FCM dengan $c=2$ dan $m=2$.

In [ ]:
pts = np.array([[1, 1], [2, 1], [4, 3], [5, 4], [6, 5]], float); lab = list("ABCDE")
mu_e, sd_e = pts.mean(0), pts.std(0)                       # sigma populasi (sama dengan StandardScaler)
Z_e = (pts - mu_e) / sd_e
tab_std = pd.DataFrame({"x1": pts[:, 0], "x2": pts[:, 1], "z1": Z_e[:, 0], "z2": Z_e[:, 1]}, index=lab)
print("Langkah 1 - Standardisasi.  Rata-rata:", mu_e, " Std populasi:", sd_e.round(4))
display(tab_std.round(4))

C_e = np.cov(Z_e.T, bias=True)
w_e, V_e = np.linalg.eigh(C_e); o = np.argsort(-w_e); w_e, V_e = w_e[o], V_e[:, o]
if V_e[:, 0].sum() < 0: V_e[:, 0] *= -1
pc1 = Z_e @ V_e[:, 0]
print(f"\nLangkah 2 - PCA.  Korelasi r = {C_e[0, 1]:.4f}")
print(f"Nilai eigen: lambda1 = {w_e[0]:.4f}, lambda2 = {w_e[1]:.4f}  (= 1 +/- r)")
print(f"Vektor eigen w1 = {V_e[:, 0].round(4)};  rasio varians PC1 = {w_e[0] / w_e.sum():.2%}")

In [ ]:
m_e = 2
U0 = np.array([[.8, .2], [.7, .3], [.5, .5], [.3, .7], [.2, .8]])
Um0 = U0 ** m_e
v1 = (Um0.T @ pc1) / Um0.sum(0)
D1 = np.abs(pc1[:, None] - v1[None, :])
inv1 = D1 ** (-2 / (m_e - 1)); U1 = inv1 / inv1.sum(1, keepdims=True)
J1 = float(((U1 ** m_e) * D1 ** 2).sum())

print("Langkah 3 - FCM iterasi pertama (c = 2, m = 2)")
display(pd.DataFrame({"Skor PC1": pc1, "u1 awal": U0[:, 0], "u2 awal": U0[:, 1], "u1^2": Um0[:, 0], "u2^2": Um0[:, 1]}, index=lab).round(4))
print(f"Pusat cluster 1: v1 = sum(u1^2 * PC1) / sum(u1^2) = {(Um0[:, 0] * pc1).sum():.4f} / {Um0[:, 0].sum():.4f} = {v1[0]:.4f}")
print(f"Pusat cluster 2: v2 = {v1[1]:.4f}")
display(pd.DataFrame({"d1": D1[:, 0], "d2": D1[:, 1], "u1 baru": U1[:, 0], "u2 baru": U1[:, 1]}, index=lab).round(4))
print(f"Contoh data C: u_C1 = 1 / [1 + (d1/d2)^2] = 1 / [1 + ({D1[2, 0]:.4f}/{D1[2, 1]:.4f})^2] = {U1[2, 0]:.4f}")
print(f"Fungsi objektif setelah iterasi pertama: J = {J1:.4f}")

# iterasi sampai konvergen
U_e = U0.copy()
for it_e in range(1, 200):
    Um = U_e ** m_e; v_e = (Um.T @ pc1) / Um.sum(0)
    D_e = np.abs(pc1[:, None] - v_e[None, :]); inv = D_e ** (-2 / (m_e - 1)); Un = inv / inv.sum(1, keepdims=True)
    if np.abs(Un - U_e).max() < 1e-5:
        U_e = Un; break
    U_e = Un
print(f"\nKonvergen pada iterasi ke-{it_e}. Pusat akhir: v1 = {v_e[0]:.4f}, v2 = {v_e[1]:.4f}")
display(pd.DataFrame({"u1 akhir": U_e[:, 0], "u2 akhir": U_e[:, 1], "Cluster": np.where(U_e[:, 0] > .5, 1, 2)}, index=lab).round(4))
print(f"Data C berkeanggotaan {U_e[2, 1]:.1%} pada cluster 2 -> data perbatasan.")

fig, ax = plt.subplots(figsize=(6.4, 2.7))
for i, l in enumerate(lab):
    ax.scatter(pc1[i], 0, s=130, color=TL if U_e[i, 0] > .5 else AM, edgecolor="black", zorder=3); ax.text(pc1[i], .12, l, ha="center", fontweight="bold")
for k_, vv in enumerate(v_e):
    ax.scatter(vv, 0, marker="X", s=210, c="black", zorder=4); ax.text(vv, -.2, f"V{k_+1}={vv:.2f}", ha="center", fontsize=8)
ax.axhline(0, color=GR, lw=.8); ax.set_ylim(-.35, .35); ax.set_yticks([]); ax.grid(False)
ax.set(xlabel="Skor PC1", title="Gambar 4.1  Data contoh pada PC1 dan pusat cluster akhir"); plt.tight_layout(); plt.show()

## 6. Standardisasi dan PCA — Evaluasi 2: Explained Variance Setelah Ekstraksi (Tabel 4.9, Gambar 4.2)
Standardisasi z-score wajib karena PCA dan FCM sensitif terhadap skala (mis. `absences` berskala 0–93, `famrel` 1–5). Jumlah komponen dipilih sebagai bilangan terkecil dengan explained variance kumulatif ≥ `VAR_THRESHOLD`.

In [ ]:
X_raw = df[FEATURES].astype(float).values
scaler = StandardScaler().fit(X_raw)
X = scaler.transform(X_raw)
print("Jumlah sampel:", X.shape[0], "| jumlah fitur:", X.shape[1])
print("Rata-rata kolom setelah z-score (maks |mu|):", np.abs(X.mean(0)).max().round(10), "| std:", X.std(0).round(3)[:3], "...")

pca_full = PCA(random_state=SEED).fit(X)
evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)
N_COMP = int(np.argmax(cum >= VAR_THRESHOLD) + 1)

evr_table = pd.DataFrame({"Explained variance": evr, "Kumulatif": cum}, index=[f"PC{i+1}" for i in range(len(evr))])
display((evr_table * 100).round(2).rename(columns=lambda c: c + " (%)"))
print(f"Jumlah komponen terpilih (kumulatif >= {VAR_THRESHOLD:.0%}) : {N_COMP}")
print(f"Total varians dipertahankan                         : {cum[N_COMP-1]:.2%}")
print(f"Varians PC1 + PC2 (visualisasi 2D)                  : {cum[1]:.2%}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6)); x_ = np.arange(1, 17)
ax[0].bar(x_, evr, color=TL); ax[0].set(title="Scree plot", xlabel="Komponen utama", ylabel="Rasio varians")
ax[1].plot(x_, cum, "o-", color=DK); ax[1].axhline(VAR_THRESHOLD, ls="--", color=AM); ax[1].axvline(N_COMP, ls="--", color=RD)
ax[1].annotate(f"{N_COMP} komponen\n({cum[N_COMP-1]:.1%})", (N_COMP, cum[N_COMP-1]), xytext=(N_COMP + 1.5, .55), arrowprops=dict(arrowstyle="->"))
ax[1].set(title="Explained variance kumulatif", xlabel="Jumlah komponen")
plt.suptitle("Gambar 4.2  Scree plot dan explained variance kumulatif", y=1.03); plt.tight_layout(); plt.show()

In [ ]:
# loading fitur pada PC1 dan PC2 (interpretasi sumbu visualisasi)
loadings = pd.DataFrame(pca_full.components_[:2].T, index=FEATURES, columns=["PC1", "PC2"])
display(loadings.round(3).sort_values("PC1", key=np.abs, ascending=False))

pca = PCA(n_components=N_COMP, random_state=SEED).fit(X)
X_pca = pca.transform(X)                                      # ruang untuk FCM
pca2 = PCA(n_components=2, random_state=SEED).fit(X)
X_vis = pca2.transform(X)                                     # ruang 2D untuk visualisasi
print("Bentuk data setelah PCA:", X_pca.shape)

## 7. Pemilihan Jumlah Cluster (Tabel 4.10, Gambar 4.3)
FCM dijalankan pada ruang PCA untuk setiap `k`. **Silhouette** dipakai untuk memilih `k` (semakin tinggi semakin baik). PC secara alami turun saat `k` naik (batas bawah $1/k$), sehingga dipakai untuk menilai ketegasan, bukan memilih `k`.

In [ ]:
rows = []
for k in range(K_MIN, K_MAX + 1):
    r_ = fcm(X_pca, k, M_FUZZ, MAX_ITER, TOL, N_INIT, SEED)
    ok = len(set(r_["labels"])) > 1
    rows.append({"k": k,
                 "Silhouette": silhouette_score(X_pca, r_["labels"]) if ok else np.nan,
                 "PC": partition_coefficient(r_["U"]), "Degenerat": is_degenerate(r_["U"])})
k_table = pd.DataFrame(rows).set_index("k")
display(k_table.round(4))

K_AUTO = int(k_table["Silhouette"].idxmax())
K = K_AUTO if K_MODE == "auto" else int(K_MANUAL)
print(f"k otomatis (Silhouette tertinggi) = {K_AUTO}  |  k yang dipakai = {K}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(k_table.index, k_table["Silhouette"], "o-", color=TL); ax[0].axvline(K, ls="--", color=RD)
ax[0].set(title="Silhouette terhadap k", xlabel="Jumlah cluster (k)", ylabel="Silhouette")
ax[1].plot(k_table.index, k_table["PC"], "s-", color=DK, label="PC"); ax[1].plot(k_table.index, 1 / k_table.index.values, ":", color=GR, label="batas 1/k")
ax[1].set(title="Partition Coefficient terhadap k", xlabel="Jumlah cluster (k)"); ax[1].legend()
plt.suptitle("Gambar 4.3  Silhouette dan Partition Coefficient terhadap k", y=1.03); plt.tight_layout(); plt.show()

## 8. Sensitivitas Parameter (Tabel 4.11–4.12, Gambar 4.4)
Pada data dengan struktur cluster lemah seperti data survei siswa ini, FCM dengan $m=2$ cenderung konvergen ke **solusi trivial**: semua pusat menyatu di rata-rata data dan setiap siswa berkeanggotaan sama ($1/c$), sehingga Partition Coefficient tepat $1/c$. Karena itu nilai bawaan $m=1{,}5$.

In [ ]:
def fit_metrics(Xp, k, m):
    r_ = fcm(Xp, k, m, MAX_ITER, TOL, N_INIT, SEED)
    return dict(Silhouette=silhouette_score(Xp, r_["labels"]), PC=partition_coefficient(r_["U"]),
                Iterasi=r_["n_iter"], Degenerat="Ya" if is_degenerate(r_["U"]) else "Tidak")

sens_m = pd.DataFrame([dict(m=m_, **fit_metrics(X_pca, 2, m_)) for m_ in [1.1, 1.3, 1.5, 1.8, 2.0, 2.5]]).set_index("m")
display(sens_m.round(4))

fig, ax = plt.subplots(figsize=(6, 3.7)); ms_ = sens_m.index.values
ax.plot(ms_, sens_m["PC"], "o-", color=DK, label="Partition Coefficient"); ax.plot(ms_, sens_m["Silhouette"], "s-", color=TL, label="Silhouette")
ax.axhline(.5, ls=":", color=GR); ax.text(2.02, .52, "batas degenerat (1/c = 0,5)", fontsize=7, color=GR)
ax.set(xlabel="Fuzzifier m", title="Gambar 4.4  Pengaruh m terhadap kualitas partisi (k = 2)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Pada m >= 2,0 Partition Coefficient = 0,5 (= 1/c): partisi degenerat dan tidak informatif.")

In [ ]:
rows = []
for v in [0.70, 0.80, 0.90, 0.95]:
    nc = int(np.argmax(cum >= v) + 1)
    Xp = PCA(n_components=nc, random_state=SEED).fit_transform(X)
    f = fit_metrics(Xp, 2, M_FUZZ)
    rows.append({"Ambang varians": f"{v:.0%}", "Jumlah komponen": nc, "Silhouette": f["Silhouette"], "PC": f["PC"]})
display(pd.DataFrame(rows).set_index("Ambang varians").round(4))
print("Ambang lebih rendah memberi Silhouette lebih tinggi tetapi membuang lebih banyak informasi; ambang 80% dipilih.")

## 9. Klasterisasi Final FCM + PCA dan Penamaan Cluster (Tabel 4.13–4.14, Gambar 4.5–4.7)
Cluster diurutkan dari rata-rata nilai (G1–G3) tertinggi ke terendah. **Tingkat prestasi** menjadi awalan nama; **sifat paling menonjol** (z-score rata-rata cluster terhadap seluruh siswa terbesar, ambang 0,4) menjadi akhiran. Dengan demikian output bukan sekadar "Cluster 0 / Cluster 1".

In [ ]:
FEATURE_LABEL = {c: FEATURE_INFO[c][0] for c in FEATURES}

def _tier_names(k):
    if k == 2: return ["Berprestasi Tinggi", "Berprestasi Rendah (Berisiko)"]
    if k == 3: return ["Berprestasi Tinggi", "Berprestasi Menengah", "Berprestasi Rendah (Berisiko)"]
    if k == 4: return ["Berprestasi Tinggi", "Menengah Atas", "Menengah Bawah", "Berprestasi Rendah (Berisiko)"]
    return ["Berprestasi Tinggi", "Menengah Atas"] + ["Menengah"] * (k - 4) + ["Menengah Bawah", "Berprestasi Rendah (Berisiko)"]

TIER_ADVICE = {
    "Berprestasi Tinggi": "Pertahankan dengan program pengayaan/akselerasi dan peran tutor sebaya.",
    "Menengah Atas": "Potensial naik ke kelompok tertinggi; dorong konsistensi belajar dan target nilai lebih tinggi.",
    "Menengah": "Perlu pendampingan belajar rutin dan monitoring perkembangan nilai antarperiode.",
    "Menengah Bawah": "Perlu bimbingan belajar tambahan dan pemantauan kehadiran serta kebiasaan belajar.",
    "Berprestasi Rendah (Berisiko)": "Prioritas intervensi: konseling akademik, remedial, kolaborasi dengan orang tua, dan pemantauan absensi.",
    "Berprestasi Menengah": "Perlu pendampingan belajar rutin dan monitoring perkembangan nilai antarperiode."}

def _traits(z):
    g = lambda cols: float(np.mean([z[c] for c in cols])); t = []
    s = g(["failures"]);              t.append((abs(s), "Riwayat Kegagalan Tinggi" if s > 0 else "Jarang Gagal Kelas"))
    s = g(["absences"]);              t.append((abs(s), "Sering Absen" if s > 0 else "Kehadiran Baik"))
    s = g(["Dalc", "Walc", "goout"]); t.append((abs(s), "Pergaulan & Konsumsi Alkohol Tinggi" if s > 0 else "Gaya Hidup Tenang"))
    s = g(["Medu", "Fedu"]);          t.append((abs(s), "Orang Tua Berpendidikan Tinggi" if s > 0 else "Orang Tua Berpendidikan Rendah"))
    s = g(["studytime"]);             t.append((abs(s), "Waktu Belajar Tinggi" if s > 0 else "Waktu Belajar Rendah"))
    s = g(["age"]);                   t.append((abs(s), "Usia Lebih Tua" if s > 0 else "Usia Lebih Muda"))
    return t

def name_clusters(df_feat, labels, k):
    """Beri nama bermakna pada tiap cluster (cluster sudah terurut dari nilai tertinggi)."""
    mu, sd = df_feat.mean(), df_feat.std().replace(0, 1); tiers = _tier_names(k); out = []
    for j in range(k):
        sub = df_feat[labels == j]; cm = sub.mean(); z = ((cm - mu) / sd).to_dict()
        traits = sorted(_traits(z), key=lambda x: -x[0])
        main = traits[0] if traits and traits[0][0] >= 0.4 else None
        tier = tiers[j]; name = f"{tier} – {main[1]}" if main else tier
        feats = sorted([(abs(v), c, v) for c, v in z.items()], reverse=True)[:4]
        ciri = [f"{FEATURE_LABEL[c]} {'lebih tinggi' if v > 0 else 'lebih rendah'} (rata-rata {cm[c]:.2f} vs {mu[c]:.2f})"
                for _, c, v in feats if abs(v) >= 0.25]
        desc = (f"{len(sub)} siswa ({len(sub)/len(df_feat):.1%}). Rata-rata nilai akhir G3 = {cm['G3']:.2f} "
                f"(seluruh siswa {mu['G3']:.2f}). " + ("Ciri menonjol: " + "; ".join(ciri) + "." if ciri else "Profil mendekati rata-rata."))
        out.append({"id": j, "nama": name, "tier": tier, "deskripsi": desc, "saran": TIER_ADVICE.get(tier, ""), "jumlah": int(len(sub))})
    seen = {}
    for o in out: seen[o["nama"]] = seen.get(o["nama"], 0) + 1
    cnt = {}
    for o in out:
        if seen[o["nama"]] > 1:
            cnt[o["nama"]] = cnt.get(o["nama"], 0) + 1; o["nama"] += f" (Kelompok {chr(64 + cnt[o['nama']])})"
    return out

# ---- FCM final pada ruang PCA
res = fcm(X_pca, K, M_FUZZ, MAX_ITER, TOL, N_INIT, SEED)
gmean = np.array([df.loc[res["labels"] == j, ["G1", "G2", "G3"]].values.mean() if (res["labels"] == j).any() else -1 for j in range(K)])
order = np.argsort(-gmean)                         # urutkan: nilai tertinggi -> terendah
U = res["U"][:, order]; centers = res["centers"][order]; labels = U.argmax(1)
cluster_info = name_clusters(df[FEATURES], labels, K); names = [c["nama"] for c in cluster_info]
print(f"FCM final: K = {K}, m = {M_FUZZ}, iterasi konvergen = {res['n_iter']}, J = {res['J']:.2f}")

tab_size = pd.DataFrame({"Nama cluster": names, "Jumlah siswa": [c["jumlah"] for c in cluster_info],
                         "Persentase": [f"{c['jumlah']/N:.1%}" for c in cluster_info]}); tab_size.index.name = "ID"
display(tab_size)
for c in cluster_info:
    display(Markdown(f"**Cluster {c['id']} - {c['nama']}**  \n{c['deskripsi']}  \n*Tindak lanjut:* {c['saran']}"))

In [ ]:
# Tabel 4.14  rata-rata fitur utama per cluster
KEYF = ["G1", "G2", "G3", "failures", "absences", "studytime", "Medu", "Fedu", "goout", "Dalc", "Walc", "age"]
profile = df[FEATURES].assign(Cluster=labels).groupby("Cluster").mean()
tab_prof = pd.DataFrame({"Keterangan": [FEATURE_LABEL[f] for f in KEYF], "Seluruh siswa": df[KEYF].mean().values}, index=KEYF)
for j in range(K): tab_prof[f"Cluster {j}"] = profile.loc[j, KEYF].values
display(tab_prof.round(2))

# Gambar 4.5  sebaran PC1-PC2
ev2 = pca2.explained_variance_ratio_; cv = pca2.transform(pca.inverse_transform(centers))
fig, ax = plt.subplots(figsize=(6.6, 5))
for j in range(K):
    p_ = X_vis[labels == j]; ax.scatter(p_[:, 0], p_[:, 1], s=20, alpha=.65, color=CL[j % 4], label=f"C{j}: {names[j]}")
ax.scatter(cv[:, 0], cv[:, 1], marker="X", s=220, c="black", edgecolor="white", label="Pusat cluster")
ax.set(xlabel=f"PC1 ({ev2[0]:.1%})", ylabel=f"PC2 ({ev2[1]:.1%})", title="Gambar 4.5  Sebaran siswa pada PC1-PC2"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Gambar 4.6  heatmap profil cluster (z-score)
mu_, sd_ = df[FEATURES].mean(), df[FEATURES].std(); zprof = ((profile - mu_) / sd_).T
fig, ax = plt.subplots(figsize=(5.2, 6)); im = ax.imshow(zprof.values, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto"); ax.grid(False)
ax.set_xticks(range(K)); ax.set_xticklabels([f"C{i}" for i in range(K)]); ax.set_yticks(range(16)); ax.set_yticklabels(FEATURES, fontsize=8)
for a in range(16):
    for b in range(K): ax.text(b, a, f"{zprof.values[a, b]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="z-score terhadap rata-rata"); ax.set_title("Gambar 4.6  Profil cluster"); plt.tight_layout(); plt.show()

# Gambar 4.7  distribusi G3 per cluster
fig, ax = plt.subplots(figsize=(6, 3.8))
for j in range(K): ax.hist(df["G3"][labels == j], bins=np.arange(-.5, 21.5, 1), alpha=.7, color=CL[j % 4], label=f"C{j}: {names[j]}")
ax.set(xlabel="Nilai akhir G3", ylabel="Jumlah siswa", title="Gambar 4.7  Distribusi G3 per cluster"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# Tabel hasil per siswa
result_df = df[FEATURES].copy(); result_df.insert(0, "Siswa", np.arange(1, N + 1))
result_df["Cluster"] = labels; result_df["Nama Cluster"] = [names[l] for l in labels]
for j in range(K): result_df[f"U{j}"] = U[:, j]
result_df["Keanggotaan Tertinggi"] = U.max(1)
print("Tabel hasil per siswa (10 baris pertama):")
display(result_df[["Siswa", "G1", "G2", "G3", "failures", "Cluster", "Nama Cluster"] + [f"U{j}" for j in range(K)] + ["Keanggotaan Tertinggi"]].head(10).round(3))

In [ ]:
# konvergensi FCM dan pusat cluster pada skala fitur asli (informasi proses, bukan metrik evaluasi)
fig, ax = plt.subplots(figsize=(5.6, 3.4)); ax.plot(range(1, len(res["hist"]) + 1), res["hist"], color=DK)
ax.set(xlabel="Iterasi", ylabel="Fungsi objektif J", title="Konvergensi FCM"); plt.tight_layout(); plt.show()
centers_orig = pd.DataFrame(scaler.inverse_transform(pca.inverse_transform(centers)), columns=FEATURES, index=[f"C{j}: {n}" for j, n in enumerate(names)])
print("Pusat cluster pada skala fitur asli:")
display(centers_orig.T.round(2))

## 10. Evaluasi 3 dan 4 — Silhouette Score dan Fuzzy Partition Coefficient (Tabel 4.15, Gambar 4.8–4.9)
Evaluasi sistem terdiri atas empat komponen. Dua komponen pertama telah dihitung pada bagian 3 (**KMO dan Bartlett**, sebelum ekstraksi) dan bagian 6 (**Explained Variance**, setelah ekstraksi). Bagian ini menghitung dua komponen terakhir pada hasil klasterisasi final lalu merangkum keempatnya.

**Silhouette Score** — kohesi dan separasi cluster: $s(i)=\dfrac{b(i)-a(i)}{\max\{a(i),b(i)\}}$, dengan $a(i)$ rata-rata jarak ke anggota cluster sendiri dan $b(i)$ rata-rata jarak terkecil ke cluster lain. Bernilai −1 sampai 1; di bawah 0,25 berarti tidak ada struktur cluster yang substansial.

**Fuzzy Partition Coefficient** — ketegasan partisi fuzzy: $PC=\dfrac1N\sum_{i=1}^{N}\sum_{j=1}^{c}u_{ij}^2$, bernilai $1/c\le PC\le 1$. $PC=1/c$ menandakan solusi degenerat (semua siswa berkeanggotaan sama). Ketegasan ternormalisasi: $(PC-1/c)/(1-1/c)$.

In [ ]:
sil_pca = silhouette_score(X_pca, labels)                 # Silhouette Score (ruang PCA tempat FCM bekerja)
sil_samples = silhouette_samples(X_pca, labels)
sil_cl = [float(sil_samples[labels == j].mean()) for j in range(K)]
PCv = partition_coefficient(U)                            # Fuzzy Partition Coefficient
PCn = (PCv - 1 / K) / (1 - 1 / K)
maxu = U.max(1); amb = float((maxu < 0.6).mean())

def sil_cat(s): return "Kuat" if s > .7 else "Cukup baik" if s > .5 else "Lemah" if s > .25 else "Lemah (< 0,25): cluster tumpang tindih"
print(f"Silhouette Score          : {sil_pca:.4f}  -> {sil_cat(sil_pca)}")
print("Silhouette per cluster    :", {f"C{j}": round(v, 3) for j, v in enumerate(sil_cl)})
print(f"Fuzzy Partition Coefficient: {PCv:.4f}  (batas bawah 1/c = {1/K:.2f}; ketegasan ternormalisasi {PCn:.0%})")
print(f"Rata-rata keanggotaan tertinggi: {maxu.mean():.3f}  |  siswa perbatasan (< 0,6): {amb:.1%}")

# Tabel 4.15  ringkasan empat komponen evaluasi
ev_sel, ev_2d = float(cum[N_COMP - 1]), float(cum[1])
ringkasan = pd.DataFrame([
    ("1. Kelayakan PCA (sebelum ekstraksi)", "KMO", f"{suit['kmo']:.3f}", f"Kategori {kmo_cat}; >= 0,6 layak"),
    ("1. Kelayakan PCA (sebelum ekstraksi)", "Bartlett chi-kuadrat", f"{suit['chi2']:.1f} (db = {suit['dof']}; p < 0,001)", "H0 ditolak: ada korelasi antar-fitur"),
    ("2. Explained variance (sesudah ekstraksi)", f"Kumulatif {N_COMP} komponen", f"{ev_sel:.2%}", f"Informasi dipertahankan; {1 - ev_sel:.1%} hilang"),
    ("2. Explained variance (sesudah ekstraksi)", "PC1 + PC2", f"{ev_2d:.2%}", "Plot 2D hanya proyeksi sebagian informasi"),
    ("3. Silhouette Score", "Ruang PCA", f"{sil_pca:.4f}", sil_cat(sil_pca)),
    ("4. Fuzzy Partition Coefficient", "PC", f"{PCv:.4f}", f"Batas bawah {1/K:.2f}; ketegasan ternormalisasi {PCn:.0%}"),
], columns=["Komponen evaluasi", "Ukuran", "Nilai", "Interpretasi"])
display(ringkasan)

In [ ]:
# Gambar 4.8  silhouette per siswa  |  Gambar 4.9  ketegasan keanggotaan fuzzy
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2)); y0 = 0
for j in range(K):
    s_ = np.sort(sil_samples[labels == j]); ax[0].fill_betweenx(np.arange(y0, y0 + len(s_)), 0, s_, color=CL[j % 4], alpha=.85, label=f"C{j}"); y0 += len(s_) + 8
ax[0].axvline(sil_pca, color=RD, ls="--"); ax[0].set(xlabel="Koefisien silhouette", title="Gambar 4.8  Silhouette per siswa"); ax[0].set_yticks([]); ax[0].legend()
sc = ax[1].scatter(X_vis[:, 0], X_vis[:, 1], c=maxu, cmap="viridis", s=22); plt.colorbar(sc, ax=ax[1], label="Derajat keanggotaan tertinggi")
ax[1].set(xlabel="PC1", ylabel="PC2", title="Gambar 4.9  Ketegasan keanggotaan fuzzy"); plt.tight_layout(); plt.show()

## 11. Prediksi Siswa Baru (Tabel 4.16)
Siswa baru melewati `scaler` dan proyeksi PCA yang **sama** dengan data latih, lalu derajat keanggotaannya dihitung terhadap pusat cluster.

In [ ]:
def predict_one(values, m=M_FUZZ):
    x = np.array([[float(values[c]) for c in FEATURES]])
    zp = pca.transform(scaler.transform(x))               # scaler & PCA data latih
    u = memberships(zp, centers, m)[0]                    # keanggotaan terhadap pusat cluster
    return u

def predict_batch(d, m=M_FUZZ):
    zp = pca.transform(scaler.transform(d[FEATURES].astype(float).values))
    Ub = memberships(zp, centers, m); lb = Ub.argmax(1)
    out = d[FEATURES].copy().reset_index(drop=True); out.insert(0, "Siswa", np.arange(1, len(out) + 1))
    out["Cluster"] = lb; out["Nama Cluster"] = [names[l] for l in lb]
    for j in range(K): out[f"U{j}"] = Ub[:, j]
    out["Keanggotaan Tertinggi"] = Ub.max(1)
    return out

top = df.loc[df["G3"].idxmax(), FEATURES].to_dict()
lowmask = (df["G3"] == 0) & (df["failures"] >= 2)
low = (df[lowmask].iloc[0] if lowmask.any() else df.loc[df["G3"].idxmin()])[FEATURES].to_dict()
mid = {c: float(v) for c, v in FORM_DEFAULT.items()}

rows = []
for desc_, v in [("Siswa dengan G3 tertinggi pada dataset", top), ("Siswa G3 = 0 dengan kegagalan kelas >= 2", low), ("Siswa dengan seluruh nilai bawaan form (profil tengah)", mid)]:
    u = predict_one(v); rows.append({"Skenario": desc_, "Keanggotaan (C0; C1)": "; ".join(f"{x:.3f}" for x in u), "Cluster prediksi": names[int(u.argmax())]})
pb = predict_batch(df)
cons = float((pb["Cluster"].values == labels).mean())
rows.append({"Skenario": f"Prediksi massal {N} siswa latih", "Keanggotaan (C0; C1)": "seluruh baris", "Cluster prediksi": f"{cons:.0%} cocok dengan label hasil pelatihan"})
display(pd.DataFrame(rows).set_index("Skenario"))

## 12. Interpretasi dan Kesimpulan (BAB IV 4.7, BAB V)

In [ ]:
c0, c1 = cluster_info[0], cluster_info[1]
display(Markdown(f"""
**Interpretasi hasil**

- Siswa terbagi menjadi **{K} kelompok**: **{c0['nama']}** ({c0['jumlah']} siswa, rata-rata G3 {profile.loc[0,'G3']:.2f}) dan **{c1['nama']}** ({c1['jumlah']} siswa, rata-rata G3 {profile.loc[1,'G3']:.2f}).
- Penanda yang paling membedakan: jumlah kegagalan kelas ({profile.loc[1,'failures']:.2f} vs {profile.loc[0,'failures']:.2f}) serta nilai periode awal G1 ({profile.loc[1,'G1']:.2f} vs {profile.loc[0,'G1']:.2f}) dan G2 ({profile.loc[1,'G2']:.2f} vs {profile.loc[0,'G2']:.2f}). Secara praktis, G1, G2, dan riwayat kegagalan adalah **penanda dini** siswa berisiko.
- Uji kelayakan PCA terpenuhi (KMO = {suit['kmo']:.3f}, {kmo_cat}; Bartlett signifikan), dan {N_COMP} komponen mempertahankan **{ev_sel:.1%}** varians.
- Partisi fuzzy cukup tegas (Fuzzy Partition Coefficient = {PCv:.3f}; ketegasan ternormalisasi {PCn:.0%}).
- Silhouette = {sil_pca:.3f} tergolong rendah: kedua kelompok saling tumpang tindih, sehingga hasil dibaca sebagai **kecenderungan**, bukan batas tegas. Sebanyak **{amb:.1%}** siswa berkeanggotaan tertinggi < 0,6 (siswa perbatasan yang sebaiknya dipantau).
- Fuzzifier m = 2 menghasilkan solusi degenerat (PC = 0,5) pada data ini, sehingga m = {M_FUZZ} dipakai.

**Keterbatasan:** tidak ada label kebenaran (evaluasi internal); nilai G1-G3 dominan pada pembentukan cluster; atribut kategorikal tidak dipakai; data hanya dua sekolah pada satu mata pelajaran (N = {N}); nama cluster adalah interpretasi dari profil rata-rata.
"""))

# ekspor hasil
result_df.to_csv("hasil_cluster_siswa.csv", index=False)
print("Hasil per siswa disimpan ke hasil_cluster_siswa.csv  (", len(result_df), "baris )")